In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install lz4 mtcnn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 106.0 MB/s eta 0:00:00


In [3]:
!pip install opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 15.9 MB/s eta 0:00:00


In [4]:
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 769.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 109.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 74.5 MB/s eta 0:00:00
  Attempting uninstall: h5py
    Found existing installation: h5py 3.16.0
    Uninstalling h5py-3.16.0:
      Successfully uninstalled h5py-3.16.0


New code

In [5]:
import os
import cv2
import numpy as np
import shutil
from tqdm import tqdm
from mtcnn import MTCNN
from google.colab import drive
from collections import defaultdict

# 2. Initialize MTCNN
detector = MTCNN()

# --- Configuration ---
# folder containing all JPEGs
input_dir = '/content/drive/MyDrive/UNIS6/cv/Downsample1'
# Destination for the 5o
drive_output_dir = '/content/drive/MyDrive/UNIS6/cv/ID_1'
os.makedirs(drive_output_dir, exist_ok=True)

def euclidean(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))

def calculate_metrics_mtcnn(image_path):
    img = cv2.imread(image_path)
    if img is None: return None
    h_img, w_img = img.shape[:2]

    if h_img < 80 or w_img < 80: return None

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    try:
        results = detector.detect_faces(img_rgb)
    except Exception:
        return None

    if not results: return None

    valid_faces = [r for r in results if r['box'][2] > 50 and r['box'][3] > 50]
    if not valid_faces: return None

    best_face = max(valid_faces, key=lambda x: x['confidence'])
    keypoints = best_face['keypoints']
    x, y, w, h = best_face['box']

    x, y = max(0, x), max(0, y)
    face_img = img[y:y+h, x:x+w]
    if face_img.size == 0: return None

    # Pose, Sharpness, and Size
    nose = keypoints['nose']
    l_eye = keypoints['left_eye']
    r_eye = keypoints['right_eye']
    pose_score = abs(euclidean(nose, l_eye) - euclidean(nose, r_eye))

    gray_face = cv2.cvtColor(face_img, cv2.COLOR_BGR2GRAY)
    sharpness_score = cv2.Laplacian(gray_face, cv2.CV_64F).var()
    size_score = (w * h) / (h_img * w_img)

    return {"pose": pose_score, "sharpness": sharpness_score, "size": size_score}

# --- Main Execution ---
if os.path.exists(input_dir):
    # 1. Group all files by subject name
    # Filename format: ID_Name_Age_Gender.jpeg
    subject_groups = defaultdict(list)
    all_files = [f for f in os.listdir(input_dir) if f.lower().endswith(('.jpeg', '.jpg'))]

    print(f"Grouping {len(all_files)} images...")
    for f in all_files:
        parts = f.split('_')
        if len(parts) >= 2:
            subject_name = parts[1] # Extracts 'MariaCallas' or 'Pele'
            subject_groups[subject_name].append(f)

    print(f"Found {len(subject_groups)} unique subjects. Starting selection...")
    found_count = 0

    # 2. Iterate through each unique subject group
    for subject_name, files in tqdm(subject_groups.items()):
        best_file_name = None
        best_full_path = None
        best_score = -float('inf')

        for f in files:
            path = os.path.join(input_dir, f)
            m = calculate_metrics_mtcnn(path)

            if m is None: continue

            # Weighted Scoring (Symmetry is key for ID photos)
            current_score = (m['size'] * 100) - (m['pose'] * 10) + (m['sharpness'] * 0.05)

            if current_score > best_score:
                best_score = current_score
                best_full_path = path
                best_file_name = f

        if best_full_path:
            # Copy with original filename (e.g., 0_MariaCallas_35_f.jpeg)
            dest_path = os.path.join(drive_output_dir, best_file_name)
            shutil.copy(best_full_path, dest_path)
            found_count += 1

    print(f"\n✅ Done! {found_count} frontal images (one per subject) saved to Drive.")
else:
    print(f"Error: Input directory {input_dir} not found.")

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Grouping 0 images...
Found 0 unique subjects. Starting selection...


0it [00:00, ?it/s]


✅ Done! 0 frontal images (one per subject) saved to Drive.


local_data_dir = '/content/drive/MyDrive/UNIS6/cv/Downsamples'
drive_output_dir = '/content/drive/MyDrive/UNIS6/cv/ID_imgs'